In [1]:
import os
import google.generativeai as genai
import sys
sys.path.append('../..')
import my_utils

import panel as pn  # GUI
pn.extension()

C:\Users\PAVAN\anaconda3\envs\MYSpace\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\PAVAN\AppData\Local\Temp\ipykernel_15780\291100148.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
# Set up Gemini API key
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [3]:
import google.generativeai as genai

models = genai.list_models()
print([m.name for m in models])

['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-lite-001', 'models/gemini-2.0-flash-lite', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-3-1b-it', 'models/gemma-3-4b-it', 'models/gemma-3-12b-it', 'models/gemma-3-27b-it', 'models/gemma-3n-e4b-it', 'models/gemma-3n-e2b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-2.5-flash-lite-preview-09-2025', 'models/gemini-3-pro-preview', 'models/gemini-3-flash-preview', 'models/gemini-3.1-pro-preview', 'models/gemini-3.1-pro-preview-customtools', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3-pro-image-preview', 'models/nano-banana-pro-preview', 'models/gemini-3.1-flash-image-preview', 'models/gemini-robotics-er-1.5-preview', 'models/gemini-2.5-computer-use-preview-10-2025', 'mod

In [4]:
def get_completion_from_messages(messages, model="gemma-3-12b-it", temperature=0.7, max_tokens=1000):
    genai.configure(api_key=os.getenv("GEMINI_API_KEY"))  

    client = genai.GenerativeModel(model_name=model)  # Initialize Gemini model

    # Remove "system" role and merge instructions into user's first message
    formatted_messages = []
    system_message = ""

    for message in messages:
        if message["role"] == "system":
            system_message = message["content"]  # Store system instruction
        elif "content" in message:  
            formatted_messages.append({
                "role": message["role"],  
                "parts": [{"text": message["content"]}]  # Correct format for Gemini
            })
        else:
            raise ValueError(f"Message missing 'content' field: {message}")  # Debugging

    # If there was a system message, merge it into the first user message
    if formatted_messages and system_message:
        formatted_messages[0]["parts"][0]["text"] = system_message + "\n\n" + formatted_messages[0]["parts"][0]["text"]

    response = client.generate_content(
        formatted_messages,  
        generation_config={
            "temperature": temperature,
            "max_output_tokens": max_tokens
        }
    )

    return response.text

In [10]:
#Function to process user messages
def process_user_message(user_input, all_messages, debug=True):
    import my_utils  # Ensure utils.py is available and correct

    delimiter = "```"

    # Step 1: Input validation
    if debug:
        print("Step 1: Input passed basic check.")

    # Step 2: Extract products and categories
    category_and_product_response = my_utils.find_category_and_product_only(
        user_input, my_utils.get_products_and_category()
    )
    print(category_and_product_response)
    category_and_product_list = my_utils.read_string_to_list(category_and_product_response)
    print(category_and_product_list)

    if debug:
        print(f"Step 2: Extracted products: {category_and_product_list}")

    # Step 3: Fetch product information
    product_information = my_utils.generate_output_string(category_and_product_list)

    if debug:
        print("Step 3: Retrieved product information.")

    # Step 4: Generate customer service response
    step_4_system_message_content = "Be concise. Only include product description and key features. Do NOT include price, rating, or extra explanation."

    messages = [
        {
            "role": "user",
            "content": step_4_system_message_content + f"\n\n{delimiter}{user_input}{delimiter}",
        },  # Merged system message
        {"role": "user", "content": f"Relevant product information:\n{product_information}"},
    ]

    final_response = get_completion_from_messages(messages)
    all_messages.append({"role": "user", "content": final_response})

    if debug:
        print("Step 4: Generated response.")

    # Step 5: Evaluate response quality
    step_6_system_message_content = """
    You are an evaluator.

    Check if the assistant response answers the user's question.
    
    Rules:
    - If ALL requested products are answered → Y
    - If ANY product is missing → N
    - Ignore wording differences
    - Be lenient
    
    Return ONLY:
    Y or N
    """

    evaluation_messages = [
        {
            "role": "user",
            "content": step_6_system_message_content
            + f"\n\nCustomer message: {delimiter}{user_input}{delimiter}\n"
            + f"Agent response: {delimiter}{final_response}{delimiter}\n"
            + f"Does the response sufficiently answer the question? reply in single word Y or N",
        }
    ]

    evaluation_response = get_completion_from_messages(evaluation_messages)

    # Ensure strict formatting
    cleaned_evaluation_response = evaluation_response.strip().upper()  # Remove spaces & enforce uppercase

    if debug:
        print(f"Step 6: Evaluation result: {cleaned_evaluation_response}")

    # Step 7: Decision based on evaluation
    if cleaned_evaluation_response == "Y":
        if debug:
            print("Step 7: Response is approved.")
        return final_response, all_messages
    else:
        if debug:
            print("Step 7: Evaluation failed, but returning response anyway.")
        return "I'm unable to provide the information you're looking for. Let me connect you with a representative.", all_messages

# Test the function
user_input = "tell me about the SmartX Pro Phone and the FotoSnap Camera, the DSLR one. Also, what tell me about your TVs?"
response, _ = process_user_message(user_input, [])
print(response)


Step 1: Input passed basic check.
```json
[
  {
    "category": "Smartphones and Accessories",
    "products": [
      "SmartX ProPhone"
    ]
  },
  {
    "category": "Cameras and Camcorders",
    "products": [
      "FotoSnap DSLR Camera"
    ]
  },
  {
    "category": "Televisions and Home Theater Systems",
    "products": [
      "CineView 4K TV",
      "SoundMax Home Theater",
      "CineView 8K TV",
      "SoundMax Soundbar",
      "CineView OLED TV"
    ]
  }
]
```
[{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone']}, {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera']}, {'category': 'Televisions and Home Theater Systems', 'products': ['CineView 4K TV', 'SoundMax Home Theater', 'CineView 8K TV', 'SoundMax Soundbar', 'CineView OLED TV']}]
Step 2: Extracted products: [{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone']}, {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera']}, {'category'

In [11]:
pn.extension(raw_css=['''
.assistant-response {
    background-color: #F6F6F6;
    padding: 10px;
    border-radius: 5px;
}
'''])

In [12]:
def collect_messages(debug=False):
    user_input = inp.value
    if debug: print(f"User Input = {user_input}")
    if user_input == "":
        return
    inp.value = ''
    global context

    response, context = process_user_message(user_input, context, debug=False)
    context.append({'role': 'assistant', 'content': f"{response}"})

    panels.append(pn.Row('User:', pn.pane.Markdown(user_input, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, css_classes=['assistant-response'])))

    return pn.Column(*panels)

In [13]:
panels = []  # collect display
context = [{'role': 'system', 'content': "You are a Service Assistant"}]

inp = pn.widgets.TextInput(placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Service Assistant")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Service Assistant')
    [2] ParamFunction(function, _pane=Str, defer_load=False, height=300, loading_indicator=True)